In [1]:
import ollama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from langchain_community.embeddings import OllamaEmbeddings

C:\Users\koush\AppData\Local\Temp\ipykernel_15512\1506778070.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import OllamaEmbeddings


In [2]:

ollama_embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434",
)

C:\Users\koush\AppData\Local\Temp\ipykernel_15512\1095601274.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  ollama_embeddings = OllamaEmbeddings(


In [4]:
from pypdf import PdfReader

file_path = "./Document Loaders in LangChain-10.pdf"  # তোমার PDF-এর path
reader = PdfReader(file_path)

print("Total pages:", len(reader.pages))

Total pages: 14


In [5]:
first_page_text = reader.pages[0].extract_text() or ""

print(first_page_text[:1000])

Document Loaders in LangChain
Code
• https://qithub.com/campusx-official/lanqchain-document-loaders/
• https://python.lanqchain.com/docs/concepts/document_loaders/
Q Retrieval-Augmented Generation (RAG)
What is RAG?
Retrieval-Augmented Generation is a powerful technique that enhances Al language 
models by:
V Combining information retrieval with language generation to produce responses 
that are both accurate and contextually relevant.
In this approach, the model first retrieves relevant documents from a knowledge base and 
then uses them as context to generate well-grounded responses.
CampusX



In [6]:
from langchain_core.documents import Document

docs = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if text.strip():
        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": file_path,
                    "page": page_number,
                },
            )
        )

print("Documents created:", len(docs))

Documents created: 14


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
)

chunks = splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Total chunks: 22


In [8]:
for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---")
    print("Metadata:", chunk.metadata)
    print("Characters:", len(chunk.page_content))
    print(chunk.page_content)


--- Chunk 1 ---
Metadata: {'source': './Document Loaders in LangChain-10.pdf', 'page': 1}
Characters: 600
Document Loaders in LangChain
Code
• https://qithub.com/campusx-official/lanqchain-document-loaders/
• https://python.lanqchain.com/docs/concepts/document_loaders/
Q Retrieval-Augmented Generation (RAG)
What is RAG?
Retrieval-Augmented Generation is a powerful technique that enhances Al language 
models by:
V Combining information retrieval with language generation to produce responses 
that are both accurate and contextually relevant.
In this approach, the model first retrieves relevant documents from a knowledge base and 
then uses them as context to generate well-grounded responses.
CampusX

--- Chunk 2 ---
Metadata: {'source': './Document Loaders in LangChain-10.pdf', 'page': 2}
Characters: 991
How RAG Works
RETRIEVAL-AUGMENTED GENERATION (RAG) WORKFLOW
USER QUERY
The parimnnal generation regent as 
the CRV LLM (LLM) users the query 
to augment relevant documents.
Search on re

In [9]:
if not chunks:
    raise ValueError("No text chunks found. Scanned PDFs may need OCR.")

texts = [
    f"search_document: {chunk.page_content}"
    for chunk in chunks
]

ollama_doc_vectors = ollama_embeddings.embed_documents(texts)

print("Total vectors:", len(ollama_doc_vectors))
print("Dimensions per vector:", len(ollama_doc_vectors[0]))

Total vectors: 22
Dimensions per vector: 768


In [10]:
query = "What is Retrieval-Augmented Generation?"

ollama_query_vector = ollama_embeddings.embed_query(
    f"search_query: {query}"
)

print("Dimensions:", len(ollama_query_vector))
print("First 5 values:", ollama_query_vector[:5])

Dimensions: 768
First 5 values: [-0.3809848129749298, 1.751736044883728, -3.9100167751312256, -2.461688756942749, 0.8434286117553711]
